In [1]:
# =============================================================================
# MODEL 3 - XGBOOST REGRESSOR (COMPLETE)
# Task: Predict Crime_Count based on Location and Date
# =============================================================================

import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
print(f"XGBoost version: {xgb.__version__}")

# =============================================================================
# 1. CONFIG
# =============================================================================
DATA_PATH = "/kaggle/input/datasets/sevindaherath/crime-per-capita-df/crime_per_capita_df_corrected.csv"
TARGET = "Crime_Count"

TRAIN_YEARS = [2020, 2021, 2022, 2023]
TEST_YEAR = 2024

MODEL_COMPARISON_PATH = "model_comparison.csv"
REPORT_TXT_PATH = "evaluation_results_xgboost.txt"
ARTIFACT_PATH = Path("artifacts") / "xgboost_model.joblib"

HOTSPOT_PERCENTS = [5, 10, 20]

FEATURES = [
    "LSOA_Latitude",
    "LSOA_Longitude",
    "LSOA_Shape_Area",
    "LSOA_Shape_Length",
    "Year",
    "Month",
    "month_sin",
    "month_cos",
    "is_summer",
    "is_winter",
    "is_spring",
    "is_autumn",
    "Total_Population",
    "pop_density",
    "income_per_capita",
    "Income_Deprivation",
    "Max_Temperature_Celsius",
    "Min_Temperature_Celsius",
    "Rainfall_mm",
    "Sunshine_Hours",
    "Air_Frost_Days",
    "crime_lag_1m",
    "crime_lag_2m",
    "crime_lag_3m",
    "crime_lag_6m",
    "crime_rolling_3m_mean",
    "crime_rolling_3m_std",
    "crime_rolling_6m_mean",
    "crime_rolling_6m_std",
    "msoa_avg_crime",
]

COMMON_XGB_PARAMS = {
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}

EXPERIMENTS = [
    {
        "label": "XGBoost - baseline",
        "params": {
            "n_estimators": 200,
            "learning_rate": 0.1,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "min_child_weight": 1,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
        },
        "fit_verbose": False,
    },
    {
        "label": "XGBoost - early stopping",
        "params": {
            "n_estimators": 1000,
            "learning_rate": 0.05,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "min_child_weight": 3,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
            "early_stopping_rounds": 50,
        },
        "fit_verbose": 100,
    },
    {
        "label": "XGBoost - shallow trees (depth=4)",
        "params": {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 4,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "min_child_weight": 3,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
        },
        "fit_verbose": False,
    },
    {
        "label": "XGBoost - deep trees (depth=8)",
        "params": {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 8,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "min_child_weight": 3,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
        },
        "fit_verbose": False,
    },
    {
        "label": "XGBoost - strong regularization",
        "params": {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "min_child_weight": 5,
            "reg_alpha": 1.0,
            "reg_lambda": 5.0,
        },
        "fit_verbose": False,
    },
    {
        "label": "XGBoost - slow LR=0.01",
        "params": {
            "n_estimators": 3000,
            "learning_rate": 0.01,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "min_child_weight": 3,
            "reg_alpha": 0.05,
            "reg_lambda": 1.0,
            "early_stopping_rounds": 100,
        },
        "fit_verbose": 200,
    },
    {
        "label": "XGBoost - wide trees depth=7 LR=0.02",
        "params": {
            "n_estimators": 1000,
            "learning_rate": 0.02,
            "max_depth": 7,
            "subsample": 0.7,
            "colsample_bytree": 0.7,
            "min_child_weight": 5,
            "reg_alpha": 0.1,
            "reg_lambda": 2.0,
            "early_stopping_rounds": 100,
        },
        "fit_verbose": 200,
    },
]

# =============================================================================
# 2. HELPERS
# =============================================================================
def safe_divide(numerator, denominator):
    denominator = float(denominator)
    if denominator == 0.0:
        return np.nan
    return float(numerator) / denominator


def round_or_nan(value, digits=4):
    if pd.isna(value) or not np.isfinite(value):
        return np.nan
    return round(float(value), digits)


def fmt4(value):
    if pd.isna(value) or not np.isfinite(value):
        return "nan"
    return f"{float(value):.4f}"


def evaluate(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'=' * 55}")
    print(f"  {model_name}")
    print(f"{'=' * 55}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  R2    : {r2:.4f}")
    print(f"  MAPE  : {mape:.2f}%")

    return {
        "Model": model_name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4),
        "MAPE": round(mape, 2),
    }


def normalize_xgb_feature_name(name, feature_list):
    if isinstance(name, str) and name.startswith("f") and name[1:].isdigit():
        idx = int(name[1:])
        if 0 <= idx < len(feature_list):
            return feature_list[idx]
    return name


# =============================================================================
# CRIME HOTSPOT EVALUATION METRICS
# PAI — Prediction Accuracy Index
# RRI — Recapture Rate Index
# PEI — Prediction Efficiency Index
# =============================================================================
def crime_hotspot_metrics(y_true, y_pred, hotspot_percent=10, model_name="Model"):
    if not (0 < hotspot_percent <= 100):
        raise ValueError("hotspot_percent must be in (0, 100].")

    df_eval = pd.DataFrame(
        {
            "actual": np.asarray(y_true, dtype=float).reshape(-1),
            "predicted": np.asarray(y_pred, dtype=float).reshape(-1),
        }
    ).dropna().reset_index(drop=True)

    if df_eval.empty:
        return {
            "Model": model_name,
            "Hotspot_%": hotspot_percent,
            "Crimes_Captured": 0,
            "Total_Crimes": 0,
            "RRI": np.nan,
            "PAI": np.nan,
            "PEI": np.nan,
        }

    threshold = np.percentile(df_eval["predicted"], 100 - hotspot_percent)
    df_eval["hotspot"] = df_eval["predicted"] >= threshold

    crimes_in_hotspots = float(df_eval.loc[df_eval["hotspot"], "actual"].sum())
    total_crimes = float(df_eval["actual"].sum())
    hotspot_area = int(df_eval["hotspot"].sum())
    total_area = int(len(df_eval))

    rri = safe_divide(crimes_in_hotspots, total_crimes)
    area_ratio = safe_divide(hotspot_area, total_area)
    pai = safe_divide(rri, area_ratio) if pd.notna(rri) and pd.notna(area_ratio) else np.nan
    pei = pai  # simplified PEI approximation

    print("\nCrime Hotspot Evaluation")
    print("-" * 40)
    print(f"Model             : {model_name}")
    print(f"Hotspot area used : top {hotspot_percent}%")
    print(f"Crimes captured   : {crimes_in_hotspots:.0f} / {total_crimes:.0f}")
    print(f"RRI               : {fmt4(rri)}")
    print(f"PAI               : {fmt4(pai)}")
    print(f"PEI               : {fmt4(pei)}")

    return {
        "Model": model_name,
        "Hotspot_%": hotspot_percent,
        "Crimes_Captured": int(round(crimes_in_hotspots)),
        "Total_Crimes": int(round(total_crimes)),
        "RRI": round_or_nan(rri, 4),
        "PAI": round_or_nan(pai, 4),
        "PEI": round_or_nan(pei, 4),
    }


def write_evaluation_report(report_path, all_results_df, hotspot_df, best_row, artifact_path):
    reg_cols = ["Model", "MAE", "RMSE", "R2", "MAPE"]
    hs_cols = ["Model", "Hotspot_%", "Crimes_Captured", "Total_Crimes", "RRI", "PAI", "PEI"]

    reg_out = all_results_df[reg_cols].copy().sort_values("MAE")
    hs_out = hotspot_df[hs_cols].copy().sort_values(["Model", "Hotspot_%"])

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# =============================================================================\n")
        f.write("# CRIME HOTSPOT EVALUATION METRICS\n")
        f.write("# PAI — Prediction Accuracy Index\n")
        f.write("# RRI — Recapture Rate Index\n")
        f.write("# PEI — Prediction Efficiency Index\n")
        f.write("# =============================================================================\n\n")
        f.write("Definitions\n")
        f.write("RRI = Crimes captured in hotspots / Total actual crimes\n")
        f.write("PAI = RRI / Hotspot area ratio\n")
        f.write("PEI = RRI / Hotspot area ratio (simplified approximation)\n\n")
        f.write("# =============================================================================\n")
        f.write("# REGRESSION EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(reg_out.to_string(index=False))
        f.write("\n\n")
        f.write("# =============================================================================\n")
        f.write("# HOTSPOT EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(hs_out.to_string(index=False))
        f.write("\n\n")
        f.write("# =============================================================================\n")
        f.write("# BEST MODEL EXPORT SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(f"Model selected : {best_row['Model']}\n")
        f.write("Metric used    : MAE\n")
        f.write(f"MAE            : {best_row['MAE']}\n")
        f.write(f"RMSE           : {best_row['RMSE']}\n")
        f.write(f"R2             : {best_row['R2']}\n")
        f.write(f"MAPE           : {best_row['MAPE']}\n")
        f.write(f"Artifact path  : {artifact_path}\n")


# =============================================================================
# 3. LOAD DATA
# =============================================================================
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape : {df.shape}")
print(f"Date range    : {df['Year'].min()} to {df['Year'].max()}")
print(f"Unique LSOAs  : {df['LSOA_Code'].nunique()}")

# =============================================================================
# 4. FEATURE VALIDATION + TIME-BASED SPLIT
# =============================================================================
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"WARNING - missing columns: {missing}")
    FEATURES = [f for f in FEATURES if f in df.columns]

print(f"\nUsing {len(FEATURES)} features -> target: '{TARGET}'")

train = df[df["Year"].isin(TRAIN_YEARS)].copy()
test = df[df["Year"] == TEST_YEAR].copy()

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print(f"Train : {len(X_train):,} rows  |  Test : {len(X_test):,} rows")

# =============================================================================
# 5. RUN XGBOOST EXPERIMENTS
# =============================================================================
results = []

for exp in EXPERIMENTS:
    label = exp["label"]
    params = {**COMMON_XGB_PARAMS, **exp["params"]}
    fit_verbose = exp["fit_verbose"]

    print(f"\n--- Fitting {label} ---")
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=fit_verbose)

    preds = np.clip(model.predict(X_test), 0, None)

    used_early_stopping = "early_stopping_rounds" in exp["params"]
    best_iter = getattr(model, "best_iteration", None)

    if used_early_stopping and best_iter is not None:
        eval_label = f"{label} (iter={best_iter})"
        print(f"Best iteration: {best_iter}")
    else:
        eval_label = label

    metrics = evaluate(eval_label, y_test, preds)

    results.append(
        {
            **metrics,
            "model_obj": model,
            "preds": preds,
            "best_iteration": best_iter if used_early_stopping else np.nan,
        }
    )

# =============================================================================
# 6. PICK BEST MODEL
# =============================================================================
best = min(results, key=lambda x: x["MAE"])
best_xgb = best["model_obj"]
best_preds = best["preds"]

print(f"\n{'=' * 55}")
print(f"Best XGBoost config : {best['Model']}")
print(f"MAE={best['MAE']}  RMSE={best['RMSE']}  R2={best['R2']}  MAPE={best['MAPE']}%")
print(f"{'=' * 55}")

# =============================================================================
# 7. FEATURE IMPORTANCE (GAIN)
# =============================================================================
print("\n--- Feature Importance - top 15 (by gain) ---")

importance_gain_raw = best_xgb.get_booster().get_score(importance_type="gain")

if importance_gain_raw:
    importance_records = []
    for feat_key, gain_val in importance_gain_raw.items():
        feat_name = normalize_xgb_feature_name(feat_key, FEATURES)
        importance_records.append({"Feature": feat_name, "Gain": gain_val})

    fi_df = (
        pd.DataFrame(importance_records)
        .groupby("Feature", as_index=False)["Gain"]
        .sum()
        .sort_values("Gain", ascending=False)
    )
else:
    fi_df = pd.DataFrame({"Feature": FEATURES, "Gain": np.zeros(len(FEATURES))})

print(fi_df.head(15).to_string(index=False))

lag_cols = [f for f in FEATURES if ("lag" in f) or ("rolling" in f) or ("msoa_avg" in f)]
lag_gain = fi_df[fi_df["Feature"].isin(lag_cols)]["Gain"].sum()
total_gain = fi_df["Gain"].sum()

lag_pct = safe_divide(lag_gain, total_gain)
other_pct = safe_divide(total_gain - lag_gain, total_gain)

print(f"\nLag/rolling features : {fmt4(lag_pct * 100 if pd.notna(lag_pct) else np.nan)}% of total gain")
print(f"All other features   : {fmt4(other_pct * 100 if pd.notna(other_pct) else np.nan)}% of total gain")

# =============================================================================
# 8. LEARNING CURVE
# =============================================================================
print("\n--- Learning Curve ---")

for frac in [0.2, 0.4, 0.6, 0.8, 1.0]:
    n_rows = max(1, int(len(X_train) * frac))
    lc_model = xgb.XGBRegressor(
        **COMMON_XGB_PARAMS,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
    )
    lc_model.fit(X_train.iloc[:n_rows], y_train.iloc[:n_rows])
    lc_preds = np.clip(lc_model.predict(X_test), 0, None)
    lc_mae = mean_absolute_error(y_test, lc_preds)
    print(f"  {int(frac * 100):3d}% of train ({n_rows:,} rows) -> MAE: {lc_mae:.4f}")

# =============================================================================
# 9. RESIDUAL ANALYSIS
# =============================================================================
print("\n--- Residual Analysis ---")

res_df = test[["LSOA_Code", "Year", "Month", TARGET]].copy()
res_df["Predicted"] = np.round(best_preds, 2)
res_df["Residual"] = res_df[TARGET] - res_df["Predicted"]
res_df["Abs_Error"] = res_df["Residual"].abs()
res_df["Pct_Error"] = res_df["Abs_Error"] / (res_df[TARGET] + 1e-9) * 100

res_df["Crime_Bucket"] = pd.cut(
    res_df[TARGET],
    bins=[0, 5, 15, 30, 50, 9999],
    labels=["0-5", "6-15", "16-30", "31-50", "50+"],
)

bucket_err = (
    res_df.groupby("Crime_Bucket", observed=True)["Abs_Error"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "Avg_MAE", "count": "N_rows"})
)
print("\nError by crime count bucket:")
print(bucket_err.to_string())

worst_lsoas = (
    res_df.groupby("LSOA_Code")["Abs_Error"]
    .mean()
    .sort_values(ascending=False)
    .head(8)
)
print("\nTop 8 hardest LSOAs:")
print(worst_lsoas.to_string())

monthly = (
    res_df.groupby("Month")["Abs_Error"]
    .mean()
    .reset_index()
    .rename(columns={"Abs_Error": "Avg_MAE"})
)
print("\nAverage error by month:")
print(monthly.to_string(index=False))

# =============================================================================
# 10. SAMPLE PREDICTIONS
# =============================================================================
print("\n--- Sample Predictions vs Actual (first 10 test rows) ---")
print(
    res_df.head(10)[["LSOA_Code", "Year", "Month", TARGET, "Predicted", "Residual"]]
    .to_string(index=False)
)

# =============================================================================
# 11. SAVE TO MODEL COMPARISON CSV
# =============================================================================
new_rows = pd.DataFrame(
    [
        {k: v for k, v in r.items() if k not in ["model_obj", "preds", "best_iteration"]}
        for r in results
    ]
)

try:
    existing = pd.read_csv(MODEL_COMPARISON_PATH)
    combined = pd.concat([existing, new_rows], ignore_index=True)
except FileNotFoundError:
    combined = new_rows

combined.to_csv(MODEL_COMPARISON_PATH, index=False)

print("\n--- Final model_comparison.csv ---")
print(combined[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))

# =============================================================================
# 12. CRIME HOTSPOT EVALUATION
# =============================================================================
print("\n--- Crime Forecasting Indices ---")

hotspot_results = []
for pct in HOTSPOT_PERCENTS:
    hs_res = crime_hotspot_metrics(
        y_true=y_test.values,
        y_pred=best_preds,
        hotspot_percent=pct,
        model_name=best["Model"],
    )
    hotspot_results.append(hs_res)

hotspot_df = pd.DataFrame(hotspot_results)

print("\nHotspot Evaluation Summary")
print(hotspot_df.to_string(index=False))

# =============================================================================
# 13. EXPORT BEST MODEL
# =============================================================================
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

best_model_package = {
    "model_name": best["Model"],
    "model": best_xgb,
    "features_used": FEATURES,
    "target": TARGET,
    "metric_used": "MAE",
    "metrics": {
        "MAE": best["MAE"],
        "RMSE": best["RMSE"],
        "R2": best["R2"],
        "MAPE": best["MAPE"],
    },
    "train_years": TRAIN_YEARS,
    "test_year": TEST_YEAR,
    "xgboost_version": xgb.__version__,
    "best_iteration": best.get("best_iteration", np.nan),
    "hotspot_percents": HOTSPOT_PERCENTS,
}

joblib.dump(best_model_package, ARTIFACT_PATH)

print("\nBest model export complete")
print(f"Model selected : {best['Model']}")
print(f"Saved artifact : {ARTIFACT_PATH}")

# =============================================================================
# 14. WRITE SINGLE TEXT REPORT (OVERWRITE EACH RUN)
# =============================================================================
all_results_df = pd.DataFrame(
    [
        {k: v for k, v in r.items() if k not in ["model_obj", "preds"]}
        for r in results
    ]
)

write_evaluation_report(
    report_path=REPORT_TXT_PATH,
    all_results_df=all_results_df,
    hotspot_df=hotspot_df,
    best_row=best,
    artifact_path=ARTIFACT_PATH,
)

print(f"\nEvaluation report saved to {REPORT_TXT_PATH}")



XGBoost version: 3.2.0
Dataset shape : (304336, 39)
Date range    : 2020 to 2024
Unique LSOAs  : 12415

Using 30 features -> target: 'Crime_Count'
Train : 241,710 rows  |  Test : 62,626 rows

--- Fitting XGBoost - baseline ---

  XGBoost - baseline
  MAE   : 4.5809
  RMSE  : 9.3732
  R2    : 0.9326
  MAPE  : 43.90%

--- Fitting XGBoost - early stopping ---
[0]	validation_0-rmse:34.68619
[100]	validation_0-rmse:10.02907
[200]	validation_0-rmse:9.76893
[300]	validation_0-rmse:9.73519
[400]	validation_0-rmse:9.72073
[421]	validation_0-rmse:9.72616
Best iteration: 371

  XGBoost - early stopping (iter=371)
  MAE   : 4.6011
  RMSE  : 9.7131
  R2    : 0.9276
  MAPE  : 43.10%

--- Fitting XGBoost - shallow trees (depth=4) ---

  XGBoost - shallow trees (depth=4)
  MAE   : 4.6680
  RMSE  : 10.2395
  R2    : 0.9195
  MAPE  : 44.59%

--- Fitting XGBoost - deep trees (depth=8) ---

  XGBoost - deep trees (depth=8)
  MAE   : 4.5636
  RMSE  : 9.4262
  R2    : 0.9318
  MAPE  : 41.84%

--- Fitting XG